# 00 — ChEMBL Target Data Store

Bu v5 sürümünde bütün kod hücreleri eğitim amacıyla satır satır açıklanmıştır. Her çalıştırılabilir satırın hemen üzerinde, o satırın görevini özetleyen kısa bir `#` notu bulunur.

Bu notebook yalnızca **veri toplama** görevini yapar.

- ChEMBL Target ID doğrulanır.
- IC50 aktiviteleri indirilir.
- Single protein format kayıtları seçilir.
- `pStandard` hesaplanır.
- Aktivite düzeyindeki ham regression CSV Google Drive'a kaydedilir.

SMILES standardizasyonu, tuz çıkarma, duplicate birleştirme ve Lipinski hesabı
bu notebookta yapılmaz. Bu işlemler yalnızca Notebook 01'de yapılır.

## 1 — Paketler

Bu bölüm, notebookun ihtiyaç duyduğu Python paketlerini kontrol eder. Eksik paketler yalnızca gerektiğinde kurulur; böylece aynı kurulum komutları gereksiz yere tekrar çalıştırılmaz. Hücre tamamlandığında sonraki bölümlerde kullanılacak temel yazılımlar hazır olur.

In [ ]:
# importlib.util paketini kullanılabilir hâle getirir.
import importlib.util
# subprocess paketini kullanılabilir hâle getirir.
import subprocess
# sys paketini kullanılabilir hâle getirir.
import sys

# `PACKAGES` değişkenine bu adımda kullanılacak değeri atar.
PACKAGES = [
    # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
    ("numpy", "numpy"),
    # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
    ("pandas", "pandas"),
    # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
    ("requests", "requests"),
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
]

# Belirtilen koleksiyondaki öğeleri sırayla işler.
for import_name, pip_name in PACKAGES:
    # Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
    if importlib.util.find_spec(import_name) is None:
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        subprocess.check_call(
            # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
            [sys.executable, "-m", "pip", "install", "-q", pip_name]
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        )

# İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
print("Paket kontrolü tamamlandı.")


## 2 — Kütüphaneler ve ayarlar

Bu bölümde kullanılacak kütüphaneler belleğe alınır ve pipeline boyunca değişmeden kullanılacak temel ayarlar tanımlanır. Target ID, klasör yolları, dosya adları ve tekrar üretilebilirlik değerleri burada merkezi olarak tutulur.

In [ ]:
# pathlib modülünden Path bileşenlerini içe aktarır.
from pathlib import Path
# json paketini kullanılabilir hâle getirir.
import json
# time paketini kullanılabilir hâle getirir.
import time

# numpy as np paketini kullanılabilir hâle getirir.
import numpy as np
# pandas as pd paketini kullanılabilir hâle getirir.
import pandas as pd
# requests paketini kullanılabilir hâle getirir.
import requests

# google.colab modülünden drive bileşenlerini içe aktarır.
from google.colab import drive
# IPython.display modülünden display bileşenlerini içe aktarır.
from IPython.display import display

# Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
drive.mount("/content/drive", force_remount=False)

# `TARGET_ID` değişkenine bu adımda kullanılacak değeri atar.
TARGET_ID = "CHEMBL206"
# `ACTIVITY_TYPE` değişkenine bu adımda kullanılacak değeri atar.
ACTIVITY_TYPE = "IC50"

# `DRIVE_DIR` değişkenine bu adımda kullanılacak değeri atar.
DRIVE_DIR = Path(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    "/content/drive/MyDrive/MOL_FET_regression_pipeline"
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)
# Gerekli klasörü mevcut değilse oluşturur.
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

# `GITHUB_OWNER` değişkenine bu adımda kullanılacak değeri atar.
GITHUB_OWNER = "MOL-FEST"
# `GITHUB_REPO` değişkenine bu adımda kullanılacak değeri atar.
GITHUB_REPO = "MOL-FET_regression"
# `GITHUB_BRANCH` değişkenine bu adımda kullanılacak değeri atar.
GITHUB_BRANCH = "main"
# `GITHUB_RAW_BASE` değişkenine bu adımda kullanılacak değeri atar.
GITHUB_RAW_BASE = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    f"https://raw.githubusercontent.com/"
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    f"{GITHUB_OWNER}/{GITHUB_REPO}/{GITHUB_BRANCH}"
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `API_BASE` değişkenine bu adımda kullanılacak değeri atar.
API_BASE = "https://www.ebi.ac.uk/chembl/api/data"
# `REQUEST_TIMEOUT` değişkenine bu adımda kullanılacak değeri atar.
REQUEST_TIMEOUT = 120
# `PAGE_LIMIT` değişkenine bu adımda kullanılacak değeri atar.
PAGE_LIMIT = 1000

# `RAW_FILENAME` değişkenine bu adımda kullanılacak değeri atar.
RAW_FILENAME = f"{TARGET_ID}_IC50_single_protein_raw.csv"
# `CONFIG_FILENAME` değişkenine bu adımda kullanılacak değeri atar.
CONFIG_FILENAME = "pipeline_config.json"

# İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
print("Target:", TARGET_ID)
# İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
print("Drive:", DRIVE_DIR)


## 3 — ChEMBL API yardımcı fonksiyonları

Bu bölüm ChEMBL servisinden veri almak veya gelen kayıtları biyolojik deney bağlamına göre düzenlemek için kullanılır. Ağ hataları, sayfalama ve eksik alanlar kontrollü biçimde ele alınır; yalnızca pipeline için gerekli kayıtlar tutulur.

In [ ]:
# `get_json` adlı yardımcı fonksiyonu tanımlar.
def get_json(url, params=None, retries=3):
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    """ChEMBL API isteğini yeniden deneme desteğiyle çalıştırır."""
    # `last_error` değişkenine bu adımda kullanılacak değeri atar.
    last_error = None

    # Belirtilen koleksiyondaki öğeleri sırayla işler.
    for attempt in range(retries):
        # Hata oluşturabilecek işlemleri güvenli bir blok içinde dener.
        try:
            # Belirtilen internet adresine HTTP GET isteği gönderir.
            response = requests.get(
                # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
                url,
                # `params` değişkenine bu adımda kullanılacak değeri atar.
                params=params,
                # `timeout` değişkenine bu adımda kullanılacak değeri atar.
                timeout=REQUEST_TIMEOUT,
            # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
            )
            # HTTP yanıtında hata kodu varsa işlemi açıklayıcı hatayla durdurur.
            response.raise_for_status()
            # Fonksiyonun ürettiği sonucu çağrıldığı yere döndürür.
            return response.json()
        # Belirtilen hata oluşursa kontrollü alternatif işlemi çalıştırır.
        except requests.RequestException as exc:
            # `last_error` değişkenine bu adımda kullanılacak değeri atar.
            last_error = exc
            # Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
            if attempt < retries - 1:
                # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
                time.sleep(2 ** attempt)

    # Koşul sağlanmadığında açıklayıcı bir hata oluşturarak işlemi durdurur.
    raise RuntimeError(f"ChEMBL API isteği başarısız: {last_error}")


# `fetch_all` adlı yardımcı fonksiyonu tanımlar.
def fetch_all(endpoint, params, record_key):
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    """Sayfalı ChEMBL endpointindeki bütün kayıtları indirir."""
    # `url` değişkenine bu adımda kullanılacak değeri atar.
    url = f"{API_BASE}/{endpoint}.json"
    # `query` değişkenine bu adımda kullanılacak değeri atar.
    query = dict(params)
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    query["limit"] = PAGE_LIMIT

    # `records` değişkenine bu adımda kullanılacak değeri atar.
    records = []

    # Koşul doğru kaldığı sürece kod bloğunu tekrarlar.
    while url:
        # `payload` değişkenine bu adımda kullanılacak değeri atar.
        payload = get_json(url, params=query)
        # Birden fazla öğeyi mevcut listeye ekler.
        records.extend(payload.get(record_key, []))

        # `next_url` değişkenine bu adımda kullanılacak değeri atar.
        next_url = payload.get("page_meta", {}).get("next")
        # `url` değişkenine bu adımda kullanılacak değeri atar.
        url = f"https://www.ebi.ac.uk{next_url}" if next_url else None
        # `query` değişkenine bu adımda kullanılacak değeri atar.
        query = None

    # Fonksiyonun ürettiği sonucu çağrıldığı yere döndürür.
    return records


# `UNIT_TO_MOLAR` değişkenine bu adımda kullanılacak değeri atar.
UNIT_TO_MOLAR = {
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "M": 1.0,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "mM": 1e-3,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "uM": 1e-6,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "µM": 1e-6,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "nM": 1e-9,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "pM": 1e-12,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
}


# `calculate_pstandard` adlı yardımcı fonksiyonu tanımlar.
def calculate_pstandard(row):
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    """Önce pChEMBL kullanır; yoksa IC50 ve birimden pStandard hesaplar."""
    # `pchembl` değişkenine bu adımda kullanılacak değeri atar.
    pchembl = pd.to_numeric(row.get("pchembl_value"), errors="coerce")
    # Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
    if pd.notna(pchembl):
        # Fonksiyonun ürettiği sonucu çağrıldığı yere döndürür.
        return float(pchembl)

    # `value` değişkenine bu adımda kullanılacak değeri atar.
    value = pd.to_numeric(row.get("standard_value"), errors="coerce")
    # `unit` değişkenine bu adımda kullanılacak değeri atar.
    unit = str(row.get("standard_units", "")).strip()

    # Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
    if pd.isna(value) or value <= 0 or unit not in UNIT_TO_MOLAR:
        # Fonksiyonun ürettiği sonucu çağrıldığı yere döndürür.
        return np.nan

    # `molar_value` değişkenine bu adımda kullanılacak değeri atar.
    molar_value = float(value) * UNIT_TO_MOLAR[unit]
    # Fonksiyonun ürettiği sonucu çağrıldığı yere döndürür.
    return float(-np.log10(molar_value))


## 4 — Target metadata

Bu bölüm pipeline'ın tek bir görevini adım adım gerçekleştirir. Kod satırlarının hemen üstündeki `#` açıklamaları, ilgili komutun ne yaptığını Python deneyimi olmayan katılımcıların takip edebileceği şekilde özetler.

In [ ]:
# `target_payload` değişkenine bu adımda kullanılacak değeri atar.
target_payload = get_json(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    f"{API_BASE}/target/{TARGET_ID}.json"
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `target_metadata` değişkenine bu adımda kullanılacak değeri atar.
target_metadata = pd.DataFrame(
    # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
    [{
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "target_chembl_id": target_payload.get("target_chembl_id"),
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "pref_name": target_payload.get("pref_name"),
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "target_type": target_payload.get("target_type"),
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "organism": target_payload.get("organism"),
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    }]
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Tabloyu notebook içinde okunabilir biçimde gösterir.
display(target_metadata)


## 5 — IC50 aktivitelerinin indirilmesi

Bu bölüm ChEMBL servisinden veri almak veya gelen kayıtları biyolojik deney bağlamına göre düzenlemek için kullanılır. Ağ hataları, sayfalama ve eksik alanlar kontrollü biçimde ele alınır; yalnızca pipeline için gerekli kayıtlar tutulur.

In [ ]:
# `activities` değişkenine bu adımda kullanılacak değeri atar.
activities = fetch_all(
    # `endpoint` değişkenine bu adımda kullanılacak değeri atar.
    endpoint="activity",
    # `params` değişkenine bu adımda kullanılacak değeri atar.
    params={
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "target_chembl_id": TARGET_ID,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "standard_type": ACTIVITY_TYPE,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "standard_relation": "=",
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    },
    # `record_key` değişkenine bu adımda kullanılacak değeri atar.
    record_key="activities",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
if not activities:
    # Koşul sağlanmadığında açıklayıcı bir hata oluşturarak işlemi durdurur.
    raise ValueError(f"{TARGET_ID} için IC50 aktivitesi bulunamadı.")

# `activity_df` değişkenine bu adımda kullanılacak değeri atar.
activity_df = pd.DataFrame(activities)
# İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
print("İndirilen aktivite:", len(activity_df))


## 6 — Single protein format kontrolü

Bu bölüm pipeline'ın tek bir görevini adım adım gerçekleştirir. Kod satırlarının hemen üstündeki `#` açıklamaları, ilgili komutun ne yaptığını Python deneyimi olmayan katılımcıların takip edebileceği şekilde özetler.

In [ ]:
# Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
if "bao_label" not in activity_df.columns:
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    activity_df["bao_label"] = pd.NA

# Eksik veya geçerli değerleri belirlemek için mantıksal maske üretir.
missing_bao = activity_df["bao_label"].isna()

# Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
if missing_bao.any() and "assay_chembl_id" in activity_df.columns:
    # `assay_ids` değişkenine bu adımda kullanılacak değeri atar.
    assay_ids = sorted(
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        activity_df.loc[missing_bao, "assay_chembl_id"]
        # Eksik kritik değerleri içeren kayıtları çıkarır.
        .dropna()
        # Veriyi belirtilen veri tipine dönüştürür.
        .astype(str)
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        .unique()
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

    # `assay_map` değişkenine bu adımda kullanılacak değeri atar.
    assay_map = {}
    # Belirtilen koleksiyondaki öğeleri sırayla işler.
    for assay_id in assay_ids:
        # `assay` değişkenine bu adımda kullanılacak değeri atar.
        assay = get_json(f"{API_BASE}/assay/{assay_id}.json")
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        assay_map[assay_id] = assay.get("bao_label")

    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    activity_df.loc[missing_bao, "bao_label"] = (
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        activity_df.loc[missing_bao, "assay_chembl_id"]
        # Veriyi belirtilen veri tipine dönüştürür.
        .astype(str)
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        .map(assay_map)
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

# `single_protein_mask` değişkenine bu adımda kullanılacak değeri atar.
single_protein_mask = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    activity_df["bao_label"]
    # Veriyi belirtilen veri tipine dönüştürür.
    .astype(str)
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    .str.lower()
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    .str.contains("single protein", na=False)
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Orijinal veriyi değiştirmemek için bağımsız bir kopya oluşturur.
activity_df = activity_df.loc[single_protein_mask].copy()

# Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
if activity_df.empty:
    # Koşul sağlanmadığında açıklayıcı bir hata oluşturarak işlemi durdurur.
    raise ValueError("Single protein format IC50 kaydı bulunamadı.")

# İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
print("Single protein format kayıt:", len(activity_df))


## 7 — SMILES ve pStandard hazırlanması

Bu bölüm moleküler yapıların kimyasal olarak tutarlı hâle getirilmesini sağlar. Geçersiz SMILES kayıtları ayrılır, tuz ve karşı iyonlar çıkarılır ve aynı parent yapıya ait tekrar ölçümler tek bir temsil altında birleştirilir.

In [ ]:
# Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
if "canonical_smiles" not in activity_df.columns:
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    activity_df["canonical_smiles"] = pd.NA

# Eksik veya geçerli değerleri belirlemek için mantıksal maske üretir.
missing_smiles = activity_df["canonical_smiles"].isna()

# Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
if missing_smiles.any():
    # `molecule_ids` değişkenine bu adımda kullanılacak değeri atar.
    molecule_ids = sorted(
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        activity_df.loc[missing_smiles, "molecule_chembl_id"]
        # Eksik kritik değerleri içeren kayıtları çıkarır.
        .dropna()
        # Veriyi belirtilen veri tipine dönüştürür.
        .astype(str)
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        .unique()
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

    # `smiles_map` değişkenine bu adımda kullanılacak değeri atar.
    smiles_map = {}
    # Belirtilen koleksiyondaki öğeleri sırayla işler.
    for molecule_id in molecule_ids:
        # `molecule` değişkenine bu adımda kullanılacak değeri atar.
        molecule = get_json(
            # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
            f"{API_BASE}/molecule/{molecule_id}.json"
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        )
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        smiles_map[molecule_id] = (
            # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
            molecule.get("molecule_structures", {}) or {}
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        ).get("canonical_smiles")

    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    activity_df.loc[missing_smiles, "canonical_smiles"] = (
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        activity_df.loc[missing_smiles, "molecule_chembl_id"]
        # Veriyi belirtilen veri tipine dönüştürür.
        .astype(str)
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        .map(smiles_map)
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

# Belirtilen fonksiyonu tablonun ilgili satır veya sütunlarına uygular.
activity_df["pStandard"] = activity_df.apply(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    calculate_pstandard,
    # `axis` değişkenine bu adımda kullanılacak değeri atar.
    axis=1,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `keep_columns` değişkenine bu adımda kullanılacak değeri atar.
keep_columns = [
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "activity_id",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "target_chembl_id",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "molecule_chembl_id",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "canonical_smiles",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "standard_value",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "standard_units",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "pchembl_value",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "pStandard",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "assay_chembl_id",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "bao_label",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
]

# `keep_columns` değişkenine bu adımda kullanılacak değeri atar.
keep_columns = [
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    column for column in keep_columns
    # Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
    if column in activity_df.columns
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
]

# `raw_output` değişkenine bu adımda kullanılacak değeri atar.
raw_output = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    activity_df[keep_columns]
    # Eksik kritik değerleri içeren kayıtları çıkarır.
    .dropna(subset=["canonical_smiles", "pStandard"])
    # DataFrame indeksini yeniden düzenler.
    .reset_index(drop=True)
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
if raw_output.empty:
    # Koşul sağlanmadığında açıklayıcı bir hata oluşturarak işlemi durdurur.
    raise ValueError("SMILES ve pStandard kontrolünden sonra veri kalmadı.")

# Tabloyu notebook içinde okunabilir biçimde gösterir.
display(raw_output.head())
# İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
print("Kaydedilecek satır:", len(raw_output))


## 8 — Google Drive kayıtları

Bu bölüm veri kaynağını hazırlar. Pipeline önce Google Drive içindeki önceki notebook çıktısını arar; dosya bulunamazsa aynı dosyayı GitHub RAW bağlantısından okumayı dener. Böylece notebooklar hem ardışık Colab çalışmasına hem de GitHub üzerinden bağımsız kullanıma uygundur.

In [ ]:
# `raw_path` değişkenine bu adımda kullanılacak değeri atar.
raw_path = DRIVE_DIR / RAW_FILENAME
# `metadata_path` değişkenine bu adımda kullanılacak değeri atar.
metadata_path = DRIVE_DIR / f"{TARGET_ID}_target_metadata.csv"
# `config_path` değişkenine bu adımda kullanılacak değeri atar.
config_path = DRIVE_DIR / CONFIG_FILENAME

# Hazırlanan tabloyu CSV dosyası olarak kaydeder.
raw_output.to_csv(raw_path, index=False)
# Hazırlanan tabloyu CSV dosyası olarak kaydeder.
target_metadata.to_csv(metadata_path, index=False)

# `config` değişkenine bu adımda kullanılacak değeri atar.
config = {
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "target_id": TARGET_ID,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "drive_dir": str(DRIVE_DIR),
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "github_raw_base": GITHUB_RAW_BASE,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "raw_filename": RAW_FILENAME,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "clean_unfiltered_filename": f"{TARGET_ID}_clean_unfiltered.csv",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "lipinski_filtered_filename": f"{TARGET_ID}_Lipinski_filtered.csv",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "feature_filename": f"{TARGET_ID}_Mordred2D_features.csv",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "manifest_filename": f"{TARGET_ID}_Mordred2D_manifest.json",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "model_filename": f"{TARGET_ID}_final_tree_model.joblib",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "imputer_filename": f"{TARGET_ID}_feature_imputer.joblib",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "bundle_filename": f"{TARGET_ID}_model_bundle.json",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
}

# Metin içeriğini belirtilen dosyaya kaydeder.
config_path.write_text(
    # Python sözlüğünü JSON metnine dönüştürür.
    json.dumps(config, ensure_ascii=False, indent=2),
    # `encoding` değişkenine bu adımda kullanılacak değeri atar.
    encoding="utf-8",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
print("Kaydedildi:", raw_path)
# İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
print("Kaydedildi:", metadata_path)
# İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
print("Kaydedildi:", config_path)
